In [1]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 0 modules, rebound 0 names.


In [2]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, quantile_loss

#alt.renderers.enable("jupyter", offline=True)
alt.renderers.enable("default")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [3]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import _style, plot_sample
from ts_uncertainty.synthetic import SyntheticGenerator

/home/richard/thesis-ts-uncertainty/.venv/lib/python3.12/site-packages/dysts/base.py:22: UserWarning: Numba not installed. Falling back to no JIT compilation.
  warnings.warn("Numba not installed. Falling back to no JIT compilation.")


In [4]:
context_length = 128
horizon_length = 64
seed = 42

sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    n_periods=1,
    n_harmonics=1,             # 1 harmonic = pure sine, no extra shape
    level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    amplitude_range=(1.0, 1.0),
    seed=seed
)
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=["Lorenz"],              # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)
inj = Injector(
    noise=False,
    shift=True,
    shift_covariate=True,
    shift_recurring_prob=0.8,
    shift_duration_range=(0.1, 0.25),
    seed=seed,
)
s = inj(sin_gen.sample())
plot_sample(s, title="Sin Sample")

alt.VConcatChart(...)

# Autoarima

In [117]:
model = ForecastModel("autoarima", season_length=1)     # or "autoarima", "chronos2", "tirex2"
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="Autoarima")

score mase: [4.04386627]
score_quantile: [0.28074908]


# Chronos-2

In [107]:
model = ForecastModel("chronos2")     # or "autoarima", "chronos2", "tirex2"
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="Chronos-2")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

score mase: [0.20473768]
score_quantile: [0.0183604]


# TiRex-2

In [6]:
model = ForecastModel("tirex2")     # or "autoarima", "chronos2", "tirex2"
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="TiRex-2")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

score mase: [0.59095285]
score_quantile: [0.04565692]


alt.VConcatChart(...)

# TimesFM-3

In [11]:
model = ForecastModel("timesfm3")
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="Timestral (123M)")

score mase: [0.2504958]
score_quantile: [0.02150606]


alt.VConcatChart(...)

# Timestral (123M)

In [10]:
model = ForecastModel("timestral", size="123M")     # 5M, 28M, 123M and 209M
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="Timestral (123M)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

score mase: [0.9173538]
score_quantile: [0.06542991]


# ci-proto

In [4]:
model = ForecastModel("ciproto")
fc = model.forecast(s)

m = fc.metric_inputs()
score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

print(f'score mase: {score_mase}')
print(f'score_quantile: {score_quantile}')

plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels, title="Timestral (123M)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

score mase: [0.12821171]
score_quantile: [0.00929949]


alt.VConcatChart(...)

# Summary all-in-one

In [5]:
models = {
    "AutoTheta": {"name": "autotheta"},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "Timestral (123M)": {'name': 'timestral', 'size': '123M'},
}

charts = []
results = []
scores_mase = []
scores_quantile = []

for key, m in models.items():
    model = ForecastModel(**m)
    fc = model.forecast(s)

    m = fc.metric_inputs()
    score_mase = mean_absolute_scaled_error(m["y_true"], m["y_pred"], m["y_train"])
    score_quantile = quantile_loss(m["y_true"], m["y_quantiles"], m["quantiles"])

    c = plot_sample(s, prediction=fc.quantiles[0], quantiles=fc.quantile_levels,
                    title=key, width=500, height=200)
    c.config = alt.Undefined          # drop the per-chart config so it can be concatenated
    charts.append(c)
    results.append(fc)
    scores_mase.append(score_mase)
    scores_quantile.append(score_quantile)

scores = pl.DataFrame({
    "model": list(models.keys()),
    "mase": [float(x.squeeze()) for x in scores_mase],
    "quantile_loss": [float(x.squeeze()) for x in scores_quantile],
})
print(scores)

grid = _style(alt.concat(*charts, columns=2, spacing=24))   # 2x2 grid, styled once
grid

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

W1107 15:55:47.163000 35110 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


ninja: no work to do.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

shape: (4, 3)
┌──────────────────┬──────────┬───────────────┐
│ model            ┆ mase     ┆ quantile_loss │
│ ---              ┆ ---      ┆ ---           │
│ str              ┆ f64      ┆ f64           │
╞══════════════════╪══════════╪═══════════════╡
│ AutoTheta        ┆ 1.264246 ┆ 0.494116      │
│ Chronos-2        ┆ 0.204738 ┆ 0.01836       │
│ TiRex-2          ┆ 0.590725 ┆ 0.04565       │
│ Timestral (123M) ┆ 0.917354 ┆ 0.06543       │
└──────────────────┴──────────┴───────────────┘


/home/richard/thesis-ts-uncertainty/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
